# Find and read records
Search for hubs, works and instances, open one, follow its links (work → instances, instance → work), and get it in the format you need. Runs against the notebook's environment: your local stack, or a deployed one (dev, stage), which it only reads. Pick it on the Notebooks page, or call `target("dev")`.

Reading needs no login.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

## Search
`type`: `all`, `hubs`, `works` or `instances`. `scope`: `all` fields or just `title`. Put a phrase in double quotes to match it exactly (no stemming). An empty `q` lists everything, newest last.

In [ ]:
import pandas as pd

QUERY = "pride and prejudice"
TYPE = "all"
SCOPE = "all"

r = bc.get("/api/search/", params={"q": QUERY, "type": TYPE, "scope": SCOPE, "limit": 20})
found = r.json()

def label(data):
    for key in ("rdfs:label", "bflc:aap"):
        if data.get(key):
            return data[key][0]
    titles = data.get("title") or []
    return titles[0].get("mainTitle", [""])[0] if titles else data.get("@id")

print(found["total"], "results;", "next page:", found["links"].get("next"))
results = pd.DataFrame([{"type": x["type"], "label": label(x["data"]), "uuid": x["uuid"], "updated": x["updated_at"]} for x in found["results"]])
results

## Open one
The first work in the results, or set `WORK_UUID` yourself.

In [ ]:
WORK_UUID = next((x["uuid"] for x in found["results"] if x["type"] == "works"), None) or bc.sample("works")
work = bc.get(f"/api/works/{WORK_UUID}.jsonld").json()
print(label(work), "·", work["@id"])
print("People see it at", f"{bc.url}/works/{WORK_UUID}")
{k: v for k, v in work.items() if k in ("@type", "title", "contribution", "language", "hasInstance", "expressionOf")}

## Follow its links
A work lists its instances (`hasInstance`) and may point at a hub (`expressionOf`).

In [ ]:
def uuid_of(uri):
    return uri.rstrip("/").rsplit("/", 1)[-1]

instances = []
for uri in work.get("hasInstance", []):
    inst = bc.get(f"/api/instances/{uuid_of(uri)}.jsonld").json()
    instances.append({"instance": label(inst), "uuid": uuid_of(uri), "types": ", ".join(inst.get("@type", []))})
pd.DataFrame(instances) if instances else "This work has no instances."

## Other formats
A suffix or an `Accept` header picks the format. `.cbd.*` is the Concise Bounded Description: the record plus the blank nodes it owns. `expand=true` pulls in the data of linked resources.

In [ ]:
for suffix in [".ttl", ".nt", ".rdf", ".cbd.jsonld", ".cbd.xml", ".vnd.sinopia.json"]:
    r = bc.get(f"/api/works/{WORK_UUID}{suffix}")
    print(f"{suffix:<18} {r.status_code}  {r.headers.get('content-type', ''):<40} {len(r.content):>7} bytes")

print()
print(bc.get(f"/api/works/{WORK_UUID}.ttl").text[:1200])

In [ ]:
expanded = bc.get(f"/api/works/{WORK_UUID}", params={"expand": "true"}, headers={"Accept": "application/ld+json"}).json()
print("Keys with expand:", sorted(expanded)[:30])

## Other resources
Authorities and other things records point at (agents, subjects, places) are stored as "other resources". Look one up by its URI, or list them.

In [ ]:
listing = bc.get("/api/resources/", params={"limit": 5}).json()
print(listing["total"], "other resources")
pd.DataFrame([{"id": x["id"], "uri": x["uri"], "label": label(x["data"]) if isinstance(x["data"], dict) else None} for x in listing["resources"]])